# ABC Vincent - 9 synthetic cases + random search

Goal:

1. Generate one synthetic population.
2. The population has one target variable `y`.
3. The population has three auxiliary variables with correlations about `0.90`, `0.80`, and `0.00` with `y`.
4. Use the same three auxiliary variables in two roles:
   - as `z` variables
   - as size variables for inclusion probabilities `pik`
5. Run all `3 z choices x 3 pik choices = 9 cases`.
6. For every case, run ABC and Random Search, both starting from Vincent/Ppi.
7. Compare both against Vincent/Ppi optimal efficiency.


## Meaning of the four final ratios

`ABC_z_over_Ppi_z`: ABC efficiency for `z` divided by Vincent/Ppi efficiency for `z`.

`ABC_y_over_Ppi_y`: ABC efficiency for `y` divided by Vincent/Ppi efficiency for `y`.

`Random_z_over_Ppi_z`: Random Search efficiency for `z` divided by Vincent/Ppi efficiency for `z`.

`Random_y_over_Ppi_y`: Random Search efficiency for `y` divided by Vincent/Ppi efficiency for `y`.

A value above `1.00` means better than Vincent/Ppi. A value near `1.00` means about equal.

In [ ]:
from pathlib import Path
import contextlib
import importlib.util
import io

import numpy as np
import pandas as pd


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for path in [start, *start.parents]:
        runner_path = path / "simulations_abc" / "terminal_running" / "run_abc_mu284_terminal.py"
        if runner_path.exists():
            return path
    raise RuntimeError("Could not find project root.")


PROJECT_ROOT = find_project_root()
RUNNER_PATH = PROJECT_ROOT / "simulations_abc" / "terminal_running" / "run_abc_mu284_terminal.py"

spec = importlib.util.spec_from_file_location("vincent_runner", RUNNER_PATH)
runner = importlib.util.module_from_spec(spec)
with contextlib.redirect_stdout(io.StringIO()):
    spec.loader.exec_module(runner)

print("Loaded ABC code from:", RUNNER_PATH)

## Settings

The values below are intentionally small so the notebook runs quickly. Increase `MAX_ITERATIONS` and `COLONY_SIZE` for a stronger search.


In [ ]:
N_UNITS = 50
N_SAMPLE = 5
CORRELATIONS = [0.90, 0.80, 0.00]

COLONY_SIZE = 5
MAX_ITERATIONS = 100
LIMIT = 5
ONLOOKER_FACTOR = 0.5

RANDOM_START_EVALS = COLONY_SIZE
RANDOM_EVALS_PER_ITERATION = int(round(COLONY_SIZE * (1.0 + ONLOOKER_FACTOR)))

INITIAL_OMEGA_VALUE = 0.0
INITIAL_RHO_VALUE = 0.5
BASE_SEED = 20260704


## Generate one population

We make one `y`, then make three auxiliary variables.

If the requested correlation is `r`, we build:

`aux = r * standardized_y + sqrt(1-r^2) * independent_noise`

Each auxiliary can be used as `z`.

For `pik`, we use the positive shifted version of the selected auxiliary, because inclusion probabilities need positive size values.


In [ ]:
def standardize(x):
    x = np.asarray(x, dtype=float)
    return (x - x.mean()) / x.std(ddof=1)


def make_positive(x, low=0.1):
    x = np.asarray(x, dtype=float)
    return x - x.min() + low


def make_population(n_units=N_UNITS, correlations=CORRELATIONS, seed=BASE_SEED):
    rng = np.random.default_rng(seed)
    location = rng.uniform(80.0, 120.0)
    scale = rng.uniform(8.0, 18.0)
    y = location + scale * rng.normal(size=n_units)
    y_std = standardize(y)

    data = {"unit": np.arange(n_units), "y": y}
    achieved = {}

    for corr in correlations:
        noise = standardize(rng.normal(size=n_units))
        aux_std = corr * y_std + np.sqrt(max(0.0, 1.0 - corr**2)) * noise
        name = f"aux_{int(round(corr * 100)):02d}"
        data[name] = aux_std
        achieved[name] = np.corrcoef(y, aux_std)[0, 1]

    return pd.DataFrame(data), achieved


df_pop, achieved = make_population()
pd.Series(achieved, name="achieved_corr").round(3)


## One function to run one case

One case means:

- choose one auxiliary variable for `z`
- choose one auxiliary variable for `pik`
- run ABC and Random Search
- compare both with Vincent/Ppi

The 9 cases are all combinations of:

`z_aux in {aux_90, aux_80, aux_00}`

`pik_aux in {aux_90, aux_80, aux_00}`


In [ ]:
def ratio(value, reference):
    return value / reference if reference > 0 and value > 0 else np.nan


def run_case(df_pop, z_aux, pik_aux, case_seed):
    y_raw = df_pop["y"].to_numpy(dtype=float)
    z_raw = df_pop[z_aux].to_numpy(dtype=float)

    # The selected pik auxiliary is shifted positive before building inclusion probabilities.
    size_raw = make_positive(df_pop[pik_aux].to_numpy(dtype=float))
    pi_raw = runner.inclusionprobabilities(size_raw, N_SAMPLE)

    sort_idx = np.argsort(z_raw / pi_raw)
    y_sorted = y_raw[sort_idx]
    z_sorted = z_raw[sort_idx]
    pi_sorted = pi_raw[sort_idx]

    N = len(df_pop)
    var_srs_y = N**2 * (1.0 - N_SAMPLE / N) * np.var(y_raw, ddof=1) / N_SAMPLE
    var_srs_z = N**2 * (1.0 - N_SAMPLE / N) * np.var(z_raw, ddof=1) / N_SAMPLE

    abc = runner.ABCAlgorithm(
        y_sorted=y_sorted,
        z_sorted=z_sorted,
        pik_sorted=pi_sorted,
        var_srs_y=var_srs_y,
        var_srs_z=var_srs_z,
        M=N_SAMPLE,
        n=N_SAMPLE,
        case_name=f"z_{z_aux}_pik_{pik_aux}",
        objective="eff_z",
        enforce_cadsd_order=True,
        random_state=case_seed,
        validation_mode="fast",
        initial_omega_value=INITIAL_OMEGA_VALUE,
        initial_rho_value=INITIAL_RHO_VALUE,
    )

    random_search = runner.RandomSearchAlgorithm(
        y_sorted=y_sorted,
        z_sorted=z_sorted,
        pik_sorted=pi_sorted,
        var_srs_y=var_srs_y,
        var_srs_z=var_srs_z,
        M=N_SAMPLE,
        n=N_SAMPLE,
        case_name=f"z_{z_aux}_pik_{pik_aux}_random",
        objective="eff_z",
        enforce_cadsd_order=True,
        random_state=case_seed + 500_000,
        validation_mode="fast",
        initial_omega_value=INITIAL_OMEGA_VALUE,
        initial_rho_value=INITIAL_RHO_VALUE,
    )

    result = abc.optimize(
        colony_size=COLONY_SIZE,
        max_iterations=MAX_ITERATIONS,
        limit=LIMIT,
        verbose=True,
        progress_interval=1,
        local_search_interval=4,
        local_search_attempts=1,
        onlooker_factor=ONLOOKER_FACTOR,
        early_stopping=False,
        min_iterations=1,
        random_searcher=random_search,
        random_start_evals=RANDOM_START_EVALS,
        random_evals_per_iteration=RANDOM_EVALS_PER_ITERATION,
    )

    return {
        "z_auxiliary": z_aux,
        "pik_auxiliary": pik_aux,
        "corr_y_z": np.corrcoef(y_raw, z_raw)[0, 1],
        "corr_y_pik_aux": np.corrcoef(y_raw, df_pop[pik_aux].to_numpy(dtype=float))[0, 1],
        "Ppi_z_eff": result["optimal_eff_z"],
        "Ppi_y_eff": result["optimal_eff_y"],
        "ABC_z_eff": result["best_eff_z"],
        "ABC_y_eff": result["best_eff_y"],
        "Random_z_eff": result["random_best_eff_z"],
        "Random_y_eff": result["random_best_eff_y"],
        "ABC_z_over_Ppi_z": ratio(result["best_eff_z"], result["optimal_eff_z"]),
        "ABC_y_over_Ppi_y": ratio(result["best_eff_y"], result["optimal_eff_y"]),
        "Random_z_over_Ppi_z": ratio(result["random_best_eff_z"], result["optimal_eff_z"]),
        "Random_y_over_Ppi_y": ratio(result["random_best_eff_y"], result["optimal_eff_y"]),
        "ABC_valid_percent": result["abc_valid_percent"],
        "Random_valid_percent": result["random_valid_percent"],
        "evaluations_ABC": result["eval_count"],
        "evaluations_Random": result["random_eval_count"],
    }


## Iteration output to watch

During each case, the optimizer prints one row per iteration.

The most important columns are:

- `ABC_z/Ppi_z`: ABC efficiency relative to Vincent/Ppi for `z`.
- `ABC_y/Ppi_y`: ABC efficiency relative to Vincent/Ppi for `y`.

`1.00` means equal to Vincent/Ppi. Above `1.00` means ABC improved on Vincent/Ppi.


## Run the 9 cases

There are 3 choices for `z` and 3 choices for `pik`, so this runs 9 cases.


In [ ]:
aux_names = ["aux_90", "aux_80", "aux_00"]
aux_seed_shift = {"aux_90": 90, "aux_80": 80, "aux_00": 0}

records = []
for z_aux in aux_names:
    for pik_aux in aux_names:
        print(f"Running z={z_aux}, pik={pik_aux}...")
        seed = BASE_SEED + 1000 * aux_seed_shift[z_aux] + aux_seed_shift[pik_aux]
        records.append(run_case(df_pop, z_aux, pik_aux, seed))

results = pd.DataFrame(records)
print("number of cases:", len(results))
results.round(4)


## Compact final table

This is the main table to read.

In [ ]:
main_cols = [
    "z_auxiliary", "pik_auxiliary", "corr_y_z", "corr_y_pik_aux",
    "ABC_z_over_Ppi_z", "ABC_y_over_Ppi_y",
    "Random_z_over_Ppi_z", "Random_y_over_Ppi_y",
]

results[main_cols].round(3)


## Average by z variable

This gives the broad pattern across the 3 inclusion-probability choices.


In [ ]:
summary_by_z = (
    results
    .groupby("z_auxiliary")[[
        "corr_y_z",
        "ABC_z_over_Ppi_z", "ABC_y_over_Ppi_y",
        "Random_z_over_Ppi_z", "Random_y_over_Ppi_y",
    ]]
    .mean()
    .round(3)
)

summary_by_z


## Save results

The CSV is stored in `artifacts/` so the notebook folder stays clean.

In [ ]:
OUT_DIR = PROJECT_ROOT / "simulations_abc" / "jupyters" / "artifacts"
OUT_DIR.mkdir(exist_ok=True)
OUT_PATH = OUT_DIR / "abc_random_9cases_synthetic.csv"
results.to_csv(OUT_PATH, index=False)
print("Saved:", OUT_PATH)
